In [1]:
import pandas as pd
import numpy as np
import scanpy as sc

In [46]:
adata = sc.read_h5ad(f'../../data/CPA_predictions/mcfarland_fold0.h5ad')


In [ ]:
adata

AnnData object with n_obs × n_vars = 196 × 19264
    obs: 'cell_type', 'condition', 'n_cells', 'fold'
    var: 'gene_name'
    layers: 'observed_mean', 'predicted_mean'

In [48]:
adatas = []
for i in range(0, 5):
    adata = sc.read_h5ad(f'../../data/CPA_predictions/mcfarland_fold{i}.h5ad')
    adatas.append(adata)

# Concatenate only the 'predicted_mean' layer matrices (assumed same variable order)
X_predicted = np.concatenate([a.layers["predicted_mean"] for a in adatas], axis=0)

# Create dataframe from combined predicted expression matrix with variable names as columns
expr_df = pd.DataFrame(X_predicted, columns=adatas[0].var_names)

# Concatenate all obs dataframes, reset index
obs_combined = pd.concat([a.obs for a in adatas], axis=0).reset_index(drop=True)

# Concatenate obs columns onto the expression dataframe
combined_df = pd.concat([expr_df, obs_combined.reset_index(drop=True)], axis=1)


In [49]:
combined_df['cell_type'] = combined_df['cell_type'].str.split('_').str[0]

In [50]:
print(combined_df.shape)
print(combined_df.head())

(991, 19268)
       A1BG      A1CF       A2M     A2ML1   A3GALT2    A4GALT     A4GNT  \
0  0.366484  0.000805  0.007192  0.010630 -0.000952  0.021795 -0.004867   
1  0.382318  0.000912  0.007356  0.010986 -0.001112  0.014284 -0.004609   
2  0.496376 -0.003765  0.009026  0.003874 -0.000886  0.144166 -0.005020   
3  0.499055 -0.003795  0.010059  0.004001 -0.001119  0.140248 -0.004847   
4  0.092001 -0.001114  0.006490  0.003736 -0.002396  0.154173 -0.009416   

       AAAS      AACS     AADAC  ...      ZXDC    ZYG11A    ZYG11B       ZYX  \
0  0.287363  0.236493 -0.007868  ...  0.066633  0.021505  0.179149  0.408325   
1  0.280561  0.236291 -0.007750  ...  0.064874  0.021192  0.177876  0.392859   
2  0.365395  0.118518 -0.004385  ...  0.064671  0.020406  0.133047  1.301345   
3  0.355447  0.120224 -0.004243  ...  0.063724  0.020207  0.133406  1.287099   
4  0.320890  0.194644 -0.009709  ...  0.057366  0.025614  0.171976  1.013641   

      ZZEF1      ZZZ3  cell_type  condition  n_cells  f

In [51]:
combined_df.to_csv('../../data/CPA_predictions/mcfarland_mean_post_all.csv', index=False)

In [52]:
pre_treatment = pd.read_csv('../../data/observed_pseudobulk/mcfarland_mean_pre_all_celllines.csv', index_col=0)

In [53]:
# To compute the LFC, we first need to:
# 1. Align combined_df and pre_treatment by cell_type and gene columns
# 2. Subtract pre_treatment's gene values from combined_df's gene values for each matching cell_type

# Merge the two dataframes on cell_type to align genes
# First, make sure 'cell_type' exists and is a column, not index, in both dataframes

# Drop columns from pre_treatment that are not gene columns (like 'condition', 'tissue' if present)
gene_cols = [col for col in pre_treatment.columns if col not in ['cell_type', 'condition', 'tissue']]

# Make sure combined_df has 'cell_type' column
assert 'cell_type' in combined_df.columns

# Only keep gene columns from both dfs for subtraction
combined_gene_df = combined_df[['cell_type'] + gene_cols].copy()
pre_gene_df = pre_treatment[['cell_type'] + gene_cols].copy()
pre_gene_df = pre_gene_df.drop_duplicates()

# Merge so that for each cell_type in combined_df, you have corresponding pre_treatment values
lfc_df = combined_gene_df.merge(pre_gene_df, on='cell_type', how='left', suffixes=('', '_pre'))

# Subtract pre-treatment gene values
for gene in gene_cols:
    lfc_df[gene] = lfc_df[gene] - lfc_df[gene + '_pre']

# Keep the LFC columns and cell_type
lfc_result = lfc_df[['cell_type'] + gene_cols]

# If you want to include other columns from combined_df (e.g. 'condition'), merge them in as needed
if 'condition' in combined_df.columns:
    lfc_result['condition'] = combined_df['condition']
if 'fold' in combined_df.columns:
    lfc_result['fold'] = combined_df['fold']

C:\Users\nbrouwer1\AppData\Local\Temp\ipykernel_21288\43137206.py:31: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  lfc_result['condition'] = combined_df['condition']
C:\Users\nbrouwer1\AppData\Local\Temp\ipykernel_21288\43137206.py:31: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  lfc_result['condition'] = combined_df['condition']
C:\Users\nbrouwer1\AppData\Local\Temp\ipykernel_21288\43137206.py:33: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, w

In [54]:
print(lfc_result.shape)
print(lfc_result.head())

(991, 19267)
  cell_type      A1BG      A1CF       A2M     A2ML1   A3GALT2    A4GALT  \
0     22RV1  0.128746 -0.019519 -0.008084  0.010630 -0.000952 -0.027412   
1     22RV1  0.144580 -0.019412 -0.007920  0.010986 -0.001112 -0.034923   
2    42MGBA -0.383271 -0.007696  0.009026  0.003874 -0.000886  0.116847   
3    42MGBA -0.380592 -0.007726  0.010059  0.004001 -0.001119  0.112928   
4      769P  0.040166 -0.021125 -0.000332  0.003736 -0.002396 -0.087257   

      A4GNT      AAAS      AACS  ...      ZXDA      ZXDB      ZXDC    ZYG11A  \
0 -0.004867 -0.039028 -0.005275  ...  0.015836  0.016136 -0.089732 -0.003297   
1 -0.004609 -0.045830 -0.005477  ...  0.017061  0.016639 -0.091490 -0.003610   
2 -0.005020  0.074930 -0.021094  ...  0.006624  0.052507 -0.044296  0.013900   
3 -0.004847  0.064981 -0.019388  ...  0.007405  0.053617 -0.045243  0.013702   
4 -0.009416  0.068059 -0.012770  ... -0.028259  0.000261 -0.000215 -0.001833   

     ZYG11B       ZYX     ZZEF1      ZZZ3  condition  f

In [55]:
lfc_result.to_csv('../../data/CPA_predictions/mcfarland_mean_LFC_all.csv', index=False)